# CLARA — Fase 2: Exploració d'Embeddings

Aquest notebook permet explorar visualment el comportament del model d'embeddings sobre el corpus legal andorrà:
- Comparar models d'embeddings multilingüe
- Visualitzar els clusters semàntics amb UMAP/t-SNE
- Testar queries per verificar la qualitat del retrieval
- Decidir el model definitiu per a la indexació (fase2_indexar.py)

> **Nota:** Un cop escollit el model, executa `fase2_indexar.py` per indexar tot el corpus.

## 0. Instal·lació de dependències

In [2]:
# Descomenta si cal instal·lar les dependències
# %pip install sentence-transformers chromadb umap-learn plotly pandas

## 1. Càrrega dels chunks

In [5]:
import json
from pathlib import Path
import pandas as pd

CHUNKS_FILE = Path('data/chunks.jsonl')

with CHUNKS_FILE.open('r', encoding='utf-8') as f:
    chunks = [json.loads(line) for line in f]

df = pd.DataFrame(chunks)
print(f'Total chunks: {len(df)}')
df.head()

Total chunks: 76035


,chunk_id,text,source_title,source_url,category,local_path,chunk_index,total_chunks,n_chars
0,b9c2bbb2-6936-4989-ab05-dbdf11dda71a,"Decret 62/2026, del 25-2-2026, pel qual s'apro...","Reglament, del 25 de febrer del 2026, del Prog...",https://portaljuridicandorra.ad/R20260225C,general,data\raw_html\2026\Reglament__del_25_de_febrer...,0,54,798
1,d854467d-799f-4408-9b2a-48bb9fa50524,Article 12. \nVigència\n\nArticle 13. \nCauses...,"Reglament, del 25 de febrer del 2026, del Prog...",https://portaljuridicandorra.ad/R20260225C,general,data\raw_html\2026\Reglament__del_25_de_febrer...,1,54,261
2,cc821686-1d0f-4c3f-95c6-c14b61bfb5af,Article 17. \nSeguiment i control\n\nExposició...,"Reglament, del 25 de febrer del 2026, del Prog...",https://portaljuridicandorra.ad/R20260225C,general,data\raw_html\2026\Reglament__del_25_de_febrer...,2,54,748
3,ef8d3f27-643c-4636-9240-8a4adb2f1dba,La convocatòria dels programes de millora de l...,"Reglament, del 25 de febrer del 2026, del Prog...",https://portaljuridicandorra.ad/R20260225C,general,data\raw_html\2026\Reglament__del_25_de_febrer...,3,54,568
4,a96ea855-cd02-4026-a5be-7c4b75bff462,"El 2025, la situació del mercat laboral ha seg...","Reglament, del 25 de febrer del 2026, del Prog...",https://portaljuridicandorra.ad/R20260225C,general,data\raw_html\2026\Reglament__del_25_de_febrer...,4,54,759


In [6]:
# Estadístiques del corpus
print('Chunks per categoria:')
print(df['category'].value_counts())
print(f'\nMida mitjana dels chunks: {df["n_chars"].mean():.0f} caràcters')
print(f'Mida mínima: {df["n_chars"].min()}')
print(f'Mida màxima: {df["n_chars"].max()}')

Chunks per categoria:
category
general    76035
Name: count, dtype: int64

Mida mitjana dels chunks: 571 caràcters
Mida mínima: 50
Mida màxima: 800


## 2. Comparativa de models d'embeddings

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODELS = {
    'MiniLM-L12 (ràpid)': 'paraphrase-multilingual-MiniLM-L12-v2',
    'E5-large (qualitat)': 'intfloat/multilingual-e5-large-instruct',
    # 'BGE-M3 (best)': 'BAAI/bge-m3',  # Descomentar si tens temps
}

# Frases de prova en català
test_sentences = [
    'La pena per robatori és de dos a sis anys de presó.',
    'El delicte de furt és sancionat amb multa.',
    'Les societats anònimes han de tenir un capital mínim.',
    'Els treballadors tenen dret a vacances pagades.',
]

results = {}
for name, model_id in MODELS.items():
    print(f'Carregant {name}...')
    model = SentenceTransformer(model_id)
    t0 = time.time()
    embs = model.encode(test_sentences, normalize_embeddings=True)
    elapsed = time.time() - t0
    results[name] = {'model': model, 'time': elapsed, 'dim': embs.shape[1]}
    print(f'  Dimensió: {embs.shape[1]} | Temps: {elapsed:.2f}s')

Carregant MiniLM-L12 (ràpid)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dimensió: 384 | Temps: 0.49s
Carregant E5-large (qualitat)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\marcs\Documents\GitHub\CLARA\.venvCLARA\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marcs\.cache\huggingface\hub\models--intfloat--multilingual-e5-large-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


... (8 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

  Dimensió: 1024 | Temps: 1.25s


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Avaluar similitud semàntica: les dues primeres frases (penal) han ser
# més similars entre si que amb les frases de dret laboral/mercantil
for name, r in results.items():
    model = r['model']
    embs = model.encode(test_sentences, normalize_embeddings=True)
    sim_matrix = cosine_similarity(embs)
    print(f'\n{name}:')
    print(f'  Similitud penal-penal   (esperat alt): {sim_matrix[0,1]:.3f}')
    print(f'  Similitud penal-laboral (esperat baix): {sim_matrix[0,3]:.3f}')


MiniLM-L12 (ràpid):
  Similitud penal-penal   (esperat alt): 0.730
  Similitud penal-laboral (esperat baix): 0.007

E5-large (qualitat):
  Similitud penal-penal   (esperat alt): 0.910
  Similitud penal-laboral (esperat baix): 0.828


## 3. Visualització de clusters semàntics (UMAP)

In [9]:
import umap
import plotly.express as px

# Usar el model triat (canvia aquí si vols provar un altre)
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(MODEL_NAME)

# Mostra aleatòria per a la visualització (UMAP és lent amb molts punts)
SAMPLE_SIZE = min(500, len(df))
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f'Generant embeddings per a {SAMPLE_SIZE} chunks...')
embeddings = model.encode(
    df_sample['text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generant embeddings per a 500 chunks...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [10]:
print('Reduint dimensions amb UMAP...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
coords_2d = reducer.fit_transform(embeddings)

df_sample['umap_x'] = coords_2d[:, 0]
df_sample['umap_y'] = coords_2d[:, 1]
df_sample['text_preview'] = df_sample['text'].str[:80] + '...'

fig = px.scatter(
    df_sample,
    x='umap_x', y='umap_y',
    color='category',
    hover_data=['source_title', 'text_preview'],
    title=f'Clusters semàntics del corpus CLARA ({SAMPLE_SIZE} chunks)',
    width=900, height=600,
)
fig.show()

Reduint dimensions amb UMAP...


c:\Users\marcs\Documents\GitHub\CLARA\.venvCLARA\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 4. Prova manual del retrieval

In [12]:
import chromadb
from chromadb.config import Settings

# Carregar la col·lecció (si ja s'ha executat fase2_indexar.py)
client = chromadb.PersistentClient(
    path='chroma_db',
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_collection('clara_legal')
print(f'Vectors indexats: {collection.count()}')

Vectors indexats: 76035


In [13]:
def test_query(query: str, n: int = 5):
    """Prova una consulta i mostra els fragments més rellevants."""
    q_emb = model.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=n,
        include=['documents', 'metadatas', 'distances']
    )
    print(f'Query: "{query}"\n')
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ), 1):
        score = 1 - dist
        print(f'[{i}] Score: {score:.2%} | {meta["title"]}')
        print(f'    {doc[:200]}...')
        print()

# Provar diverses consultes
test_query('Quina és la pena per al delicte de robatori amb violència?')

Query: "Quina és la pena per al delicte de robatori amb violència?"

[1] Score: 77.32% | Llei 9/2005, del 21 de febrer, qualificada del Codi penal
    Article 485
 

Sense contingut

Article 486. 
Violació de propietat privada

1. Les conductes previstes als articles 194 i 195 realitzades amb violència o intimidació o força en les coses en altres bé...

[2] Score: 76.04% | Llei 28/2022, del 14 de juliol, qualificada de protecció civil
    Article 91. 
Sancions

1. Les infraccions molt greus són objecte de la sanció de multa no inferior a 10.001 euros i no superior a 50.000 euros.

2. Les infraccions greus són objecte de la sanció de mu...

[3] Score: 75.16% | Llei 9/2005, del 21 de febrer, qualificada del Codi penal
    Article 198. 
Furt qualificat
 

1. El delicte de furt ha de ser castigat amb pena de presó de tres mesos a tres anys si hi concorre alguna de les circumstàncies següents:

a) Que la cosa sostreta tin...

[4] Score: 74.96% | Llei 2/2013, del 18 d’abril, de la funció de 

In [14]:
test_query('Requisits per constituir una societat de responsabilitat limitada')

Query: "Requisits per constituir una societat de responsabilitat limitada"

[1] Score: 75.76% | Reglament, del 27 de març del 2024, que desenvolupa la cotització a la Caixa Andorrana de Seguretat Social de les persones que treballen per compte propi
    Article 4. 
Els socis de societats

1. Els socis de les societats que no estiguin donades de baixa provisional al Registre tributari poden cotitzar com a persones que realitzen una activitat per compt...

[2] Score: 70.25% | Llei 7/2021, del 29 d’abril, de recuperació i de resolució d’entitats bancàries i d’empreses d’inversió
    q) En relació amb la societat de gestió d'actius i sense perjudici de l'aplicació de la resta de lletres:

- Dur a terme activitats alienes al seu objecte social que posin en perill la consecució dels...

[3] Score: 70.08% | Llei 6/2014, del 24 d’abril, de serveis socials i sociosanitaris
    4. Els comuns han de promoure i facilitar l'accés a les prestacions tècniques de la seva competència dins els límits de

In [15]:
test_query('Drets dels treballadors en cas de malaltia')

Query: "Drets dels treballadors en cas de malaltia"

[1] Score: 71.35% | Llei 31/2018, del 6 de desembre, de relacions laborals
    Article 49. 
Seguretat i salut en el treball

1. La persona assalariada té dret a una protecció eficaç en matèria de seguretat i salut en el treball i, ensems, té l'obligació de realitzar la seva tasc...

[2] Score: 70.22% | Llei 17/2008, del 3 d’octubre, de la seguretat social
    Article 117. 
Prestacions per a les persones assalariades
 

1. Les persones assalariades tenen dret a totes les prestacions previstes en aquesta branca quan, a més dels requisits particulars exigits ...

[3] Score: 68.88% | Llei 9/2004, del 27 de maig, de la funció pública de l’Administració de Justícia (LFPAJ)
    Article 55. 
Salut laboral i prevenció de riscs

1. El Consell Superior de la Justícia promou la millora de les condicions de treball del seu personal, orientada a prevenir, evitar o disminuir la poss...

[4] Score: 67.24% | Reglament, de l'1 de desembre del 2010, de

## 5. Conclusió

Anota aquí les teves observacions sobre els models provats:

| Model | Qualitat retrieval | Velocitat CPU | Decisió |
|---|---|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | ⭐⭐⭐ | ⚡⚡⚡ | ✅ Escollit |
| `multilingual-e5-large-instruct` | ⭐⭐⭐⭐ | ⚡⚡ | 🔄 Alternativa |

Un cop escollit el model, actualitza `EMBED_MODEL` a `fase2_indexar.py` i executa:
```bash
python fase2_indexar.py
```